# Unit 5, Lecture 1: Testing an agent

A normal function gives the same answer every time, so you `assert` and move on.
An agent calls a model, which can answer differently each time, so the naive test
`assert agent.answer() == "billing"` **fails randomly** even when nothing is wrong.

The fix is not to give up on testing. It is to **split the agent in two**:

1. The **skeleton** (tools, routing, state) is deterministic. Test it **exactly**.
2. The **model-dependent** part needs a **fake model** (a test double) to test offline.

The quiet truth of this whole course: every workflow ran offline in its tests.
That was this lecture, applied from day one. **This entire notebook runs with no
lane and no tokens.**

## 1. The skeleton tests exactly

Your tools and logic are plain functions. Same input, same output. They test the
ordinary way, and this is **most** of your system.

In [ ]:
from cse476.testing_agents import check_refund_eligible, summarise_findings

# exact tests, no model, no flakiness
assert check_refund_eligible(10, 2000.0) is True     # within limits
assert check_refund_eligible(40, 2000.0) is False    # too old
assert check_refund_eligible(10, 9000.0) is False    # too large
assert summarise_findings(["a", "b"]) == "a; b"
assert summarise_findings([]) == "no findings"
print("all skeleton tests passed, exactly, offline")

## 2. The seam: take the client as an argument

`decide_route` depends on a model. But it takes the client as an **argument**,
not a global. That argument is a **seam**: a place to swap the real model for a
fake. This one habit is what makes model-dependent code testable.

In [ ]:
import inspect
from cse476.testing_agents import decide_route

print(inspect.getsource(decide_route))

## 3. The fake model: you control what it says

A `FakeClient` returns a canned answer **you** choose. That lets you test your
defensive logic against every awkward thing a real model might do, instantly and
for free. A fake is not a worse real model; for testing it is **better**, because
you control it.

In [ ]:
from cse476.testing_agents import decide_route, FakeClient

# force every awkward answer a real model might give:
assert decide_route("x", FakeClient("billing"))      == "billing"    # clean
assert decide_route("x", FakeClient("  ACCOUNT  "))  == "account"    # messy case+spaces
assert decide_route("x", FakeClient("technical"))    == "technical"  # clean
assert decide_route("x", FakeClient("banana"))       == "general"    # garbage -> fallback
assert decide_route("x", FakeClient(""))             == "general"    # empty -> fallback
print("defensive logic tested against every edge case, offline")

## 4. A recording fake tests what the agent SENT

Half of agent bugs are on the way **out**: a malformed prompt, an extra call. A
fake that records what it was asked (a **spy**) catches those, which a real model
never could.

In [ ]:
spy = FakeClient("billing")
decide_route("my card was double charged", spy)

assert len(spy.prompts_seen) == 1                      # asked exactly once
assert "double charged" in spy.prompts_seen[0]         # sent the ticket
assert "classify" in spy.prompts_seen[0].lower()       # asked for a classification
print("we asserted the QUESTION, not just the answer")
print("the agent sent:", repr(spy.prompts_seen[0]))

## Debugging: testing finds THAT it broke, debugging finds WHERE

A billing ticket got routed to `general`. Something is wrong, but where? An agent
has **three layers** a bug can hide in, and the whole skill is telling which one
broke instead of guessing:

1. the **prompt** you sent,
2. the **answer** the model gave,
3. the **logic** that handled it.

The recipe: **reproduce, isolate, fix, lock in.**

### Step 1: reproduce it (turn a flaky bug into a certain one)

A model is non-deterministic, so a bug seen once may vanish next run. Capture the
exact answer that caused it and replay it forever with `ReplayClient`.

In [ ]:
from cse476.testing_agents import ReplayClient, decide_route

# the model once answered "Billing." with a trailing period, which caused a miss
buggy = ReplayClient(["Billing."])
print("route:", decide_route("refund please", buggy))   # -> "general", every time

# it reproduces identically, offline, with no model
print("deterministic:",
      decide_route("x", ReplayClient(["Billing."])) == decide_route("x", ReplayClient(["Billing."])))

### Step 2: isolate the layer

`diagnose_route` returns all three layers side by side, so you can see which one
broke instead of guessing.

In [ ]:
from cse476.testing_agents import diagnose_route

d = diagnose_route("refund please", ReplayClient(["Billing."]))
for layer, value in d.items():
    print(f"{layer:12}: {value!r}")
print()
print("Diagnosis: the model said 'Billing.' (fine), but our logic did an exact")
print("match and the trailing period broke it -> fell to general. LAYER 3 (logic).")
print("Fix: strip punctuation before matching. Not the prompt, not the model.")

### Step 3 and 4: fix only the fault, then lock it in

The fix is small and targeted (strip punctuation). And the replayed case becomes
a **permanent test**, so this exact bug can never return silently. A bug you fix
without a test is a bug that comes back.

In [ ]:
# the ReplayClient(["Billing."]) case is now a regression test:
def test_trailing_period_still_routes_billing():
    # once we fix decide_route to strip punctuation, this will pass
    from cse476.testing_agents import decide_route
    # (today it returns "general"; after the fix it should return "billing")
    result = decide_route("x", ReplayClient(["Billing."]))
    print("current behaviour:", result, "(this is the test that locks in the fix)")

test_trailing_period_still_routes_billing()

### Bonus: test that a tool FAILURE is handled

A tool can raise in production, and that must not crash the whole run. `safe_call`
turns a raised exception into data, so you can **test the failure path**.

In [ ]:
from cse476.testing_agents import safe_call

def broken_tool(x):
    raise ValueError("service down")

# the failure path, now testable offline:
assert safe_call(broken_tool, "in")["ok"] is False        # no crash
assert "service down" in safe_call(broken_tool, "in")["error"]
assert safe_call(lambda x: x.upper(), "hi") == {"ok": True, "value": "HI"}
print("the broken tool did not crash the agent; the failure became data")

### The debugging recipe

In [ ]:
from cse476.testing_agents import DEBUGGING_MAP, the_debugging_recipe

for concept, meaning in DEBUGGING_MAP.items():
    print(f"{concept:22} ->  {meaning}")
print()
for step, rule in the_debugging_recipe().items():
    print(f"{step:14}: {rule}")

## The testing pyramid

In [ ]:
from cse476.testing_agents import TESTING_MAP, the_testing_pyramid

for concept, meaning in TESTING_MAP.items():
    print(f"{concept:22} ->  {meaning}")
print()
for tier, rule in the_testing_pyramid().items():
    print(f"{tier:14}: {rule}")

Push your tests **down** the pyramid: lots of cheap exact tests at the bottom,
a few costly real-model ones at the top. The lower a test sits, the faster it
runs and the more you trust it. A suite that is mostly fast exact tests gets run
on every commit; one that is mostly slow flaky real-model calls gets muted and
protects nothing.

## Your turn

**1. Test a tool exactly.** Pick one tool from your capstone. Write three exact
tests: a normal case, an edge case, and a failing case.

**2. Fake a model.** Take one piece of model-dependent logic, add a client seam,
then write a `FakeClient` that returns a messy answer (wrong case, extra spaces).
Test that your logic cleans it.

**3. Spot the flaky test.** Find one place you were tempted to assert the model
said an exact sentence. Rewrite it as a fake-model test or a loose check. That is
moving a test down the pyramid.

In [ ]:
# your work here
